In [ ]:
!pip install pdfplumber textstat transformers scikit-learn nltk --quiet
import pdfplumber

def extract_text_from_pdf(file_path):
    with pdfplumber.open(file_path) as pdf:
        full_text = "\n".join([
            page.extract_text()
            for page in pdf.pages
            if page.extract_text()
        ])
    return full_text


In [ ]:
from google.colab import files

uploaded = files.upload()

# Extract file path
pdf_path = list(uploaded.keys())[0]
print(f"📄 File uploaded: {pdf_path}")


Saving Products and Circular economy (EU).pdf to Products and Circular economy (EU).pdf
📄 File uploaded: Products and Circular economy (EU).pdf


In [ ]:
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)  # replaces multiple spaces/tabs/newlines with one space
    text = re.sub(r'[^a-zA-Z0-9 .,]', '', text)  # removes weird characters
    return text.strip()


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from transformers import pipeline
import nltk
from nltk.tokenize import sent_tokenize # Import sent_tokenize


nltk.download('punkt')
nltk.download('punkt_tab')

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
def summarize_text(text):
    max_chunk = 1024
    sentences = sent_tokenize(text)
    current_chunk = ""
    chunks = []

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chunk:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    summaries = []
    for chunk in chunks:
        input_length = len(chunk.split())
        max_len = min(130, int(input_length * 0.8),summarizer.model.config.max_position_embeddings - 2)  # cap output to 80% of input
        max_len = max(30, max_len)

    summaries = summarizer(chunks, max_length=80, min_length=30, do_sample=False)
    return " ".join([s['summary_text'] for s in summaries])


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Device set to use cpu


In [ ]:
import textstat
from nltk.tokenize import word_tokenize

def get_readability_metrics(text):
    return {
        "readability": textstat.flesch_reading_ease(text),
        "avg_sentence_length": textstat.avg_sentence_length(text),
        "lexical_diversity": len(set(word_tokenize(text))) / len(word_tokenize(text))
    }


In [ ]:
# Extract text from uploaded file
raw_text = extract_text_from_pdf(pdf_path)
cleaned_text = clean_text(raw_text)

# Limit input for summarization
limited_text = " ".join(cleaned_text.split()[:800])
summary = summarize_text(limited_text)
input_length = len(limited_text.split())
max_len = min(100, int(input_length * 0.7))
min_len = max(20, int(max_len * 0.6))

summary = summarizer(limited_text, max_length=max_len, min_length=min_len, do_sample=False)[0]['summary_text']

print("📝 Shortened Summary:\n")
print(summary)


# Get readability scores
metrics = get_readability_metrics(cleaned_text)


Your max_length is set to 80, but your input_length is only 45. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=22)


IndexError: index out of range in self

In [ ]:
print("\n📊 Readability Metrics:")
for key, value in metrics.items():
    print(f"{key.replace('_', ' ').title()}: {value:.2f}")



📊 Readability Metrics:
Readability: 17.74
Avg Sentence Length: 19.60
Lexical Diversity: 0.23
